# Week 6 - KL Divergence for 1D Gaussians

**Mode**: experiment  
**Companion note**: `notes/w06_where_kl_appears.md`  
**Workflow**: this notebook is a TODO scaffold. The `_solved` copy starts identical; do your work there and keep this one as the clean baseline.

## Goal

Use the Gaussian, expectation, variance, and transformation tools from Weeks 2-6 to understand KL divergence as a measure of distribution mismatch.

## What you will do by hand

1. Derive the analytic KL divergence between two 1D Gaussians.
2. Implement your derived formula.
3. Compare analytic KL with Monte Carlo estimates.
4. Compare analytic KL with histogram-based estimates.
5. Explain why empirical distribution matching gets fragile.

No closed-form KL solution is written in this scaffold. Fill the TODOs yourself.

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=6)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["font.size"] = 12

In [ ]:
config = {
    "seed": 6,
    "n_grid": 800,
    "n_samples": 20_000,
    "p": {"mu": 0.0, "sigma": 1.0},
    "q": {"mu": 1.0, "sigma": 1.4},
}
config

---

## Section 1: The Objects

Let

$$
p(x) = N(\mu_p, \sigma_p^2),
\qquad
q(x) = N(\mu_q, \sigma_q^2).
$$

KL divergence is defined as

$$
D_{KL}(p\|q)
=
\mathbb{E}_{X\sim p}\left[\log p(X)-\log q(X)\right].
$$

The expectation is under $p$, not under $q$. This direction matters.

In [ ]:
def normal_pdf(x, mu, sigma):
    """Evaluate the 1D Gaussian PDF."""
    x = np.asarray(x)
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))


def normal_logpdf(x, mu, sigma):
    """Evaluate the 1D Gaussian log-density."""
    x = np.asarray(x)
    return -np.log(sigma * np.sqrt(2 * np.pi)) - 0.5 * ((x - mu) / sigma) ** 2


# Lightweight checks for the helper functions.
x_grid = np.linspace(-6, 6, config["n_grid"])
p_pdf = normal_pdf(x_grid, config["p"]["mu"], config["p"]["sigma"])
q_pdf = normal_pdf(x_grid, config["q"]["mu"], config["q"]["sigma"])

assert np.all(p_pdf >= 0)
assert np.all(q_pdf >= 0)
assert np.isfinite(normal_logpdf(x_grid, 0.0, 1.0)).all()
print("Helper checks passed.")

In [ ]:
fig, ax = plt.subplots()
ax.plot(x_grid, p_pdf, label="p: source distribution")
ax.plot(x_grid, q_pdf, label="q: target/approx distribution")
ax.set_title("Two 1D Gaussian densities")
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.legend()
plt.show()

---

## Section 2: TODO 1 - Derive KL for 1D Gaussians by Hand

Start from

$$
D_{KL}(p\|q)
=
\mathbb{E}_{p}\left[\log p(X)-\log q(X)\right].
$$

Use your Week 3 Gaussian log-density:

$$
\log N(x;\mu,\sigma^2)
=
-\log(\sigma\sqrt{2\pi})
-
\frac{(x-\mu)^2}{2\sigma^2}.
$$

### TODO 1.1

Write $\log p(X)$ and $\log q(X)$ explicitly.

Your work:

$$
\log p(X)=\text{TODO}
$$

$$
\log q(X)=\text{TODO}
$$

### TODO 1.2

Substitute those into $\mathbb{E}_p[\log p(X)-\log q(X)]$.

Your work:

$$
D_{KL}(p\|q)=\text{TODO}
$$

### TODO 1.3

Use these facts under $X\sim p$:

$$
\mathbb{E}_p[X-\mu_p]=0,
\qquad
\mathbb{E}_p[(X-\mu_p)^2]=\sigma_p^2.
$$

You will also need this quantity:

$$
\mathbb{E}_p[(X-\mu_q)^2].
$$

Hint: rewrite

$$
X-\mu_q=(X-\mu_p)+(\mu_p-\mu_q).
$$

Your work:

$$
\mathbb{E}_p[(X-\mu_q)^2]=\text{TODO}
$$

### TODO 1.4

Simplify everything into one final analytic expression.

Your final formula:

$$
D_{KL}(p\|q)=\text{TODO}
$$

---

## Section 3: TODO 2 - Implement the Analytic KL

Implement the formula you derived above. Keep this cell as the source of truth for the rest of the notebook.

In [ ]:
def kl_gaussian_1d(mu_p, sigma_p, mu_q, sigma_q):
    """Return D_KL(N(mu_p, sigma_p^2) || N(mu_q, sigma_q^2))."""
    # TODO: implement your formula from Section 2.
    kl = np.nan
    return float(kl)


same_kl = kl_gaussian_1d(0.0, 1.0, 0.0, 1.0)
shift_kl = kl_gaussian_1d(0.0, 1.0, 1.0, 1.0)
wide_kl = kl_gaussian_1d(0.0, 1.0, 0.0, 2.0)

{"same": same_kl, "shifted_mean": shift_kl, "wider_q": wide_kl}

# Optional self-checks after completing the TODO:
# assert np.isclose(same_kl, 0.0)
# assert shift_kl > same_kl
# assert wide_kl > same_kl

---

## Section 4: TODO 3 - Monte Carlo KL Estimate

Monte Carlo estimate:

1. Sample $X_1,\dots,X_n\sim p$.
2. Compute $\log p(X_i)-\log q(X_i)$ for each sample.
3. Average those values.

This estimates the same expectation in the KL definition.

In [ ]:
def monte_carlo_kl(mu_p, sigma_p, mu_q, sigma_q, n_samples, rng):
    """Estimate D_KL(p || q) by sampling from p."""
    samples = rng.normal(loc=mu_p, scale=sigma_p, size=n_samples)

    # TODO: compute log p(samples) - log q(samples), then average.
    estimate = np.nan
    return float(estimate)


sample_sizes = [100, 1_000, 10_000, 100_000]
mc_rows = []
for n in sample_sizes:
    estimate = monte_carlo_kl(
        config["p"]["mu"],
        config["p"]["sigma"],
        config["q"]["mu"],
        config["q"]["sigma"],
        n,
        rng,
    )
    mc_rows.append({"n_samples": n, "mc_kl": estimate})

mc_rows

# Optional self-check after completing TODO 2 and TODO 3:
# analytic = kl_gaussian_1d(config["p"]["mu"], config["p"]["sigma"], config["q"]["mu"], config["q"]["sigma"])
# assert abs(mc_rows[-1]["mc_kl"] - analytic) < 0.03

---

## Section 5: TODO 4 - Histogram-Based KL Estimate

Histogram KL turns samples into approximate discrete distributions.

This is intentionally fragile. Your job is to see how much the estimate depends on binning and smoothing.

Use common bins for samples from $p$ and $q$. Then estimate probabilities per bin and compute the discrete KL sum.

In [ ]:
samples_p = rng.normal(config["p"]["mu"], config["p"]["sigma"], size=config["n_samples"])
samples_q = rng.normal(config["q"]["mu"], config["q"]["sigma"], size=config["n_samples"])


def histogram_kl(samples_p, samples_q, bins, eps=1e-12):
    """Estimate KL from two sample sets using shared histogram bins."""
    counts_p, _ = np.histogram(samples_p, bins=bins)
    counts_q, _ = np.histogram(samples_q, bins=bins)

    # TODO: convert counts to probability vectors.
    p_hat = np.full_like(counts_p, np.nan, dtype=float)
    q_hat = np.full_like(counts_q, np.nan, dtype=float)

    # TODO: add eps smoothing if needed, renormalize, and compute sum p_hat * log(p_hat / q_hat).
    estimate = np.nan
    return float(estimate)


bins = np.linspace(-6, 6, 51)
hist_kl = histogram_kl(samples_p, samples_q, bins)
hist_kl

# Optional self-check after completing the TODO:
# assert np.isfinite(hist_kl)
# assert hist_kl >= 0

---

## Section 6: TODO 5 - Sensitivity to Binning

Compare histogram KL estimates for several bin counts. The point is not to find the perfect bin count; the point is to notice that empirical distribution estimates have knobs.

In [ ]:
bin_counts = [20, 50, 100, 200]
hist_rows = []

for count in bin_counts:
    bins = np.linspace(-6, 6, count + 1)
    estimate = histogram_kl(samples_p, samples_q, bins)
    hist_rows.append({"bin_count": count, "hist_kl": estimate})

hist_rows

In [ ]:
# TODO: after TODO 2 and TODO 5 are complete, plot analytic KL vs histogram estimates.
# analytic = kl_gaussian_1d(config["p"]["mu"], config["p"]["sigma"], config["q"]["mu"], config["q"]["sigma"])
# xs = [row["bin_count"] for row in hist_rows]
# ys = [row["hist_kl"] for row in hist_rows]
#
# fig, ax = plt.subplots()
# ax.axhline(analytic, color="black", linestyle="--", label="analytic KL")
# ax.plot(xs, ys, marker="o", label="histogram KL")
# ax.set_xscale("log")
# ax.set_xlabel("number of bins")
# ax.set_ylabel("KL estimate")
# ax.set_title("Histogram KL sensitivity to binning")
# ax.legend()
# plt.show()

---

## Section 7: TODO 6 - Reflection

Answer these in your own words after the code works.

1. Why is $D_{KL}(p\|q)$ not usually the same as $D_{KL}(q\|p)$?
2. Which estimate was more stable: Monte Carlo KL or histogram KL? Why?
3. Why would histogram KL become much harder in high dimensions?
4. How does this connect to the idea of matching a simple Gaussian distribution to a complicated data distribution?

Your reflection:

- TODO:
- TODO:
- TODO:
- TODO:

---

## Takeaways

Fill this section after solving the TODOs.

- Analytic result:
- Empirical evidence:
- Main pitfall:
- Connection to diffusion / flow matching: